In [ ]:
import wandb
from src.training.model_trainer import ModelTrainer
from src.preprocessing.experiment_preprocessor import ExperimentPreprocessor
from src.training.objectives import LogisticRegressionObjective, MlpObjective
from src.common.constants import Constants as consts
from src.common.wandb_config import WANDB_ENTITY, WANDB_PROJECT
from src.training.artifact_manager import ArtifactManager
from src.common.experiment_configs import ExperimentPreprocessConfig, ExperimentInfo

In [10]:
from pprint import pprint

fft_real_tts_preprocess_config = ExperimentPreprocessConfig(
    splits_names=["train", "dev"],
    fraction=0.05,
    use_audio_id_sampling=False,
    use_standardize=True,
    balance_splits_strategy=None,
    remove_by_query='starting_point > 36.0 and config.str.contains("vocoders")',
).get_dict()

fft_real_vocoders_preprocess_config = ExperimentPreprocessConfig(
    splits_names=["train", "dev"],
    fraction=0.05,
    use_audio_id_sampling=False,
    use_standardize=True,
    balance_splits_strategy=None,
    remove_by_query='starting_point > 36.0 and config.str.contains("tts")',
).get_dict()

experiment_info = ExperimentInfo(
    experiment_name="domain_shift_2",
    models=["MLP"],
    description="Testing domain shift on MLP model with FFT features",
    experiment_preprocess_configs={
        "real_tts": fft_real_tts_preprocess_config,
        "real_vocoders": fft_real_vocoders_preprocess_config,
    },
    n_trials=1,
    objective_params={"epochs": 2, "use_pos_weight": True},
).get_dict()

pprint(experiment_info)

{'description': 'Testing domain shift on MLP model with FFT features',
 'experiment_name': 'domain_shift_2',
 'experiment_preprocess_configs': {'real_tts': {'balance_splits_strategy': None,
                                                'fraction': 0.05,
                                                'remove_by_query': 'starting_point '
                                                                   '> 36.0 and '
                                                                   'config.str.contains("vocoders")',
                                                'splits_names': ['train',
                                                                 'dev'],
                                                'use_audio_id_sampling': False,
                                                'use_standardize': True},
                                   'real_vocoders': {'balance_splits_strategy': None,
                                                     'fraction': 0.05,
                  

In [7]:
fft_preprocessor = ExperimentPreprocessor(feat_suffix=consts.fft_emb_suffix)
print("Preprocessing FFT features...")


fft_real_tts_dataset_map = fft_preprocessor.preprocess_data(**fft_real_tts_preprocess_config)
fft_dataloaders_map = fft_preprocessor.get_dataloaders(fft_real_tts_dataset_map, batch_size=32)
print("Dataloader keys:", fft_dataloaders_map.keys())

fft_real_vocoders_dataset_map = fft_preprocessor.preprocess_data(**fft_real_vocoders_preprocess_config)
fft_dataloaders_map_vocoders = fft_preprocessor.get_dataloaders(fft_real_vocoders_dataset_map, batch_size=32)
print("Dataloader keys:", fft_dataloaders_map_vocoders.keys())

2026-05-13 17:44:08 | INFO     | FeatureLoader | Using FFT embeddings suffix
2026-05-13 17:44:08 | INFO     | FeatureLoader | Constructed file path: /Users/mikolajkarapka/Projects/audio-deepfake-detection-uwr/data/collected_data/feature_extracted_fft.npy
2026-05-13 17:44:08 | INFO     | FeatureLoader | Constructed file path: /Users/mikolajkarapka/Projects/audio-deepfake-detection-uwr/data/collected_data/feature_extracted.csv
2026-05-13 17:44:08 | INFO     | ExperimentPreprocessor | ExperimentPreprocessor initialized with device: mps and feature suffix: '_fft'
2026-05-13 17:44:08 | INFO     | FeatureLoader | Constructed file path: /Users/mikolajkarapka/Projects/audio-deepfake-detection-uwr/data/collected_data/splited_data/feature_extracted_train.csv
2026-05-13 17:44:08 | INFO     | FeatureLoader | Loading metadata from /Users/mikolajkarapka/Projects/audio-deepfake-detection-uwr/data/collected_data/splited_data/feature_extracted_train.csv


Preprocessing FFT features...


2026-05-13 17:44:19 | INFO     | ExperimentPreprocessor | Removing records from split 'train' using query: starting_point > 36.0 and config.str.contains("vocoders")
2026-05-13 17:44:21 | INFO     | ExperimentPreprocessor | Sampling 5.0% of data for split 'train'...
2026-05-13 17:44:22 | INFO     | ExperimentPreprocessor | Standardizing features for split 'train'...
2026-05-13 17:44:22 | INFO     | FeatureLoader | Constructed file path: /Users/mikolajkarapka/Projects/audio-deepfake-detection-uwr/data/collected_data/splited_data/feature_extracted_dev.csv
2026-05-13 17:44:22 | INFO     | FeatureLoader | Loading metadata from /Users/mikolajkarapka/Projects/audio-deepfake-detection-uwr/data/collected_data/splited_data/feature_extracted_dev.csv
2026-05-13 17:44:25 | INFO     | ExperimentPreprocessor | Removing records from split 'dev' using query: starting_point > 36.0 and config.str.contains("vocoders")
2026-05-13 17:44:25 | INFO     | ExperimentPreprocessor | Sampling 5.0% of data for spli

Dataloader keys: dict_keys(['train', 'dev'])


2026-05-13 17:44:35 | INFO     | ExperimentPreprocessor | Removing records from split 'train' using query: starting_point > 36.0 and config.str.contains("tts")
2026-05-13 17:44:36 | INFO     | ExperimentPreprocessor | Sampling 5.0% of data for split 'train'...
2026-05-13 17:44:36 | INFO     | ExperimentPreprocessor | Standardizing features for split 'train'...
2026-05-13 17:44:37 | INFO     | FeatureLoader | Constructed file path: /Users/mikolajkarapka/Projects/audio-deepfake-detection-uwr/data/collected_data/splited_data/feature_extracted_dev.csv
2026-05-13 17:44:37 | INFO     | FeatureLoader | Loading metadata from /Users/mikolajkarapka/Projects/audio-deepfake-detection-uwr/data/collected_data/splited_data/feature_extracted_dev.csv
2026-05-13 17:44:39 | INFO     | ExperimentPreprocessor | Removing records from split 'dev' using query: starting_point > 36.0 and config.str.contains("tts")
2026-05-13 17:44:39 | INFO     | ExperimentPreprocessor | Sampling 5.0% of data for split 'dev'...

Dataloader keys: dict_keys(['train', 'dev'])


In [8]:
dataloaders_dict = {
    "fft_real_tts": fft_dataloaders_map,
    "fft_real_vocoders": fft_dataloaders_map_vocoders,
}

In [11]:
run = wandb.init(
    project=WANDB_PROJECT,
    entity=WANDB_ENTITY,
    name=experiment_info["experiment_name"],
    config=experiment_info,
)

model_trainer = ModelTrainer()
artifact_manager = ArtifactManager(experiment_name=experiment_info["experiment_name"])

objectives = [MlpObjective]
objectives_params = experiment_info["objective_params"]
n_trials = experiment_info["n_trials"]

for config_key, dataloaders_map in dataloaders_dict.items():
    train_dataloader = dataloaders_map["train"]
    dev_dataloader = dataloaders_map["dev"]
    for objective_cls in objectives:
        print(f"Training {objective_cls.__name__} with {config_key} config...")
        objective = objective_cls(train_loader=train_dataloader, val_loader=dev_dataloader)
        best_params, best_value = model_trainer.optuna_train(
            objective=objective,
            n_trials=n_trials,
            **objectives_params
        )
        print(f"Best params for {objective_cls.__name__} with {config_key} config: {best_params}, best value: {best_value}")
        file_name = f"{objective_cls.__name__}_{config_key}_best_params"
        artifact_manager.save_params(
            params=best_params,
            file_name=file_name,
        )
        params_file_path = artifact_manager.get_params_file_path(file_name=file_name)
        run.log_artifact(params_file_path)
        print(f"Saved best params for {objective_cls.__name__} with {config_key} config to {params_file_path} and logged to wandb.")

run.finish()


2026-05-13 17:53:04 | INFO     | ModelTrainer | Starting Optuna hyperparameter optimization...
[I 2026-05-13 17:53:04,749] A new study created in memory with name: no-name-71a86518-4449-4387-a830-6233954cc055


Training MlpObjective with fft_real_tts config...


  0%|          | 0/1 [00:00<?, ?it/s]

2026-05-13 17:53:04 | INFO     | audio_deepfake.MlpClassifier | Using device: mps
2026-05-13 17:53:25 | INFO     | MlpObjective | Epoch 1/2 - Train Loss: 0.4432, Train Acc: 0.8862, Val Loss: 0.2778, Val Acc: 0.9584, EER: 0.0646
2026-05-13 17:53:47 | INFO     | MlpObjective | Epoch 2/2 - Train Loss: 0.2456, Train Acc: 0.9400, Val Loss: 0.2199, Val Acc: 0.9538, EER: 0.0516
2026-05-13 17:53:47 | INFO     | ModelTrainer | Best trial: 0
2026-05-13 17:53:47 | INFO     | ModelTrainer | Best parameters: {'dropout_rate': 0.11859236549778439, 'n_layers': 3, 'hidden_size_0': 768, 'hidden_size_1': 704, 'hidden_size_2': 544, 'lr': 0.00011935928744667846, 'weight_decay': 0.009294862211673065, 'pos_weight': 11.253008972441044}
2026-05-13 17:53:47 | INFO     | ModelTrainer | Best value: 0.05164271220564842
2026-05-13 17:53:47 | INFO     | ArtifactManager | Created directory for experiment: /Users/mikolajkarapka/Projects/audio-deepfake-detection-uwr/data/params/domain_shift_2
2026-05-13 17:53:47 | INFO

[I 2026-05-13 17:53:47,171] Trial 0 finished with value: 0.05164271220564842 and parameters: {'dropout_rate': 0.11859236549778439, 'n_layers': 3, 'hidden_size_0': 768, 'hidden_size_1': 704, 'hidden_size_2': 544, 'lr': 0.00011935928744667846, 'weight_decay': 0.009294862211673065, 'pos_weight': 11.253008972441044}. Best is trial 0 with value: 0.05164271220564842.
Best params for MlpObjective with fft_real_tts config: {'dropout_rate': 0.11859236549778439, 'n_layers': 3, 'hidden_size_0': 768, 'hidden_size_1': 704, 'hidden_size_2': 544, 'lr': 0.00011935928744667846, 'weight_decay': 0.009294862211673065, 'pos_weight': 11.253008972441044}, best value: 0.05164271220564842


2026-05-13 17:53:47 | INFO     | ModelTrainer | Starting Optuna hyperparameter optimization...
[I 2026-05-13 17:53:47,875] A new study created in memory with name: no-name-6006179d-d4e4-4770-8cbb-527b38a8a9e3


Saved best params for MlpObjective with fft_real_tts config to /Users/mikolajkarapka/Projects/audio-deepfake-detection-uwr/data/params/domain_shift_2/MlpObjective_fft_real_tts_best_params.pkl and logged to wandb.
Training MlpObjective with fft_real_vocoders config...


  0%|          | 0/1 [00:00<?, ?it/s]

2026-05-13 17:53:47 | INFO     | audio_deepfake.MlpClassifier | Using device: mps
2026-05-13 17:54:06 | INFO     | MlpObjective | Epoch 1/2 - Train Loss: 0.9908, Train Acc: 0.6647, Val Loss: 0.7854, Val Acc: 0.8666, EER: 0.1285
2026-05-13 17:54:24 | INFO     | MlpObjective | Epoch 2/2 - Train Loss: 0.5685, Train Acc: 0.8696, Val Loss: 0.5325, Val Acc: 0.9156, EER: 0.0795
2026-05-13 17:54:24 | INFO     | ModelTrainer | Best trial: 0
2026-05-13 17:54:24 | INFO     | ModelTrainer | Best parameters: {'dropout_rate': 0.4140788635973918, 'n_layers': 1, 'hidden_size_0': 256, 'lr': 0.00011174552464430594, 'weight_decay': 0.00017117817638983215, 'pos_weight': 30.578051241791243}
2026-05-13 17:54:24 | INFO     | ModelTrainer | Best value: 0.07951346039772034
2026-05-13 17:54:24 | INFO     | ArtifactManager | Saving params to /Users/mikolajkarapka/Projects/audio-deepfake-detection-uwr/data/params/domain_shift_2/MlpObjective_fft_real_vocoders_best_params.pkl
2026-05-13 17:54:24 | INFO     | Artifa

[I 2026-05-13 17:54:24,366] Trial 0 finished with value: 0.07951346039772034 and parameters: {'dropout_rate': 0.4140788635973918, 'n_layers': 1, 'hidden_size_0': 256, 'lr': 0.00011174552464430594, 'weight_decay': 0.00017117817638983215, 'pos_weight': 30.578051241791243}. Best is trial 0 with value: 0.07951346039772034.
Best params for MlpObjective with fft_real_vocoders config: {'dropout_rate': 0.4140788635973918, 'n_layers': 1, 'hidden_size_0': 256, 'lr': 0.00011174552464430594, 'weight_decay': 0.00017117817638983215, 'pos_weight': 30.578051241791243}, best value: 0.07951346039772034
Saved best params for MlpObjective with fft_real_vocoders config to /Users/mikolajkarapka/Projects/audio-deepfake-detection-uwr/data/params/domain_shift_2/MlpObjective_fft_real_vocoders_best_params.pkl and logged to wandb.
